In [4]:
#library imports
import torch
import torch.nn as nn
import torch.nn.functional as func
import torch.optim as optim
import torch.utils.data as data

from torchvision import transforms, datasets

import matplotlib.pyplot as plt
import numpy as np
import PIL
import urllib
import time
import os

In [ ]:
#mount drive
from google.colab import drive
drive.mount('/content/drive')

In [17]:
#git
%cd /content/drive/MyDrive/Colab\ Notebooks/Emotion_Classifier_Project
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"
!git add emotion_classifier.ipynb
!git commit -m "CNN architecture built"
!git push -u origin master

/content/drive/MyDrive/Colab Notebooks/Emotion_Classifier_Project
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@928df085d0c2.(none)')
Branch 'master' set up to track remote branch 'master' from 'origin'.
Everything up-to-date


In [ ]:
#unzip dataset to VM
!unzip '/content/drive/MyDrive/Colab Notebooks/Emotion_Classifier_Project/fer2013.zip' -d '/root/datasets'

#paths for dataset
train_path = '/root/datasets/train'
val_path = '/root/datasets/test'

In [ ]:
#DATASET LOADING

#hyperparameters
batch_size = 64;

#tranforms
transform = transforms.Compose([
    transforms.Resize(48),
    transforms.Grayscale(),
    transforms.RandomHorizontalFlip(), #data augmentation
    transforms.ToTensor(),
    transforms.Normalize(0.5077, 0.2120) #values found once below
])

#datasets
train_data = datasets.ImageFolder(train_path, transform=transform)
val_data = datasets.ImageFolder(val_path, transform=transform)

#dataloaders
train_loader = data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = data.DataLoader(val_data, batch_size=batch_size, shuffle=False)

#calculate mean & std for train set before normalized
#obtained mean = 0.5077, std = 0.2120

#mean = 0.0
#std = 0.0
#total_images = 0
#loop = 0
#for images,_ in train_loader:
#   batch = images.size(0)
#   images = images.view(batch, 1, (48*48))
#    mean += images.mean(2).sum(0)
#   std += images.std(2).sum(0)
#    total_images += batch
#mean /= total_images
#std /= total_images
#print(mean); #0.5077
#print(std); #0.2120

#sanity check
print(len(train_data))  #28709
print(len(val_data))   #7178
print(train_data.classes)
images, labels = next(iter(train_loader))
print(images.shape)   #torch.Size([64, 1, 48, 48])
print(labels.shape)   #torch.Size([64])
print(labels)
print(train_data.class_to_idx) #{'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}


In [ ]:
#dimension change calculator

W = 8 #input dimension
K = 2 #kernel size
P = 0 #padding
S = 2 #stride

out_d = ((W - K + (2*P))/S) + 1
print(out_d)

In [14]:
#CNN ARCHITECTURE
class emotion_classifier(nn.Module):
  def __init__(self):
    super(emotion_classifier, self).__init__()

    self.conv1a = nn.Conv2d(1, 16, 3, padding=1) #in channel, out channel, kernel, padding
    self.conv2a = nn.Conv2d(16, 32, 3, padding=1)
    self.conv3a = nn.Conv2d(32, 64, 3, padding=1)
    self.conv4a = nn.Conv2d(64, 128, 3, padding=1)
    self.conv5a = nn.Conv2d(128, 256, 3, padding=1)

    self.conv1b = nn.Conv2d(16, 16, 3, padding=1)
    self.conv2b = nn.Conv2d(32, 32, 3, padding=1)
    self.conv3b = nn.Conv2d(64, 64, 3, padding=1)
    self.conv4b = nn.Conv2d(128, 128, 3, padding=1)
    self.conv5b = nn.Conv2d(256, 256, 3, padding=1)

    self.pool = nn.MaxPool2d(2, 2) #kernel, stride
    self.pool2 = nn.AdaptiveAvgPool2d(1)

    self.fc1 = nn.Linear(256, 128)
    self.fc2 = nn.Linear(128, 64)
    self.fc3 = nn.Linear(64, 7)

  def forward(self, x):

    #convolution layers
    x = func.relu(self.conv1a(x))
    x = func.relu(self.conv1b(x))
    x = self.pool(x) #batchx16x24x24
    x = func.relu(self.conv2a(x))
    x = func.relu(self.conv2b(x))
    x = self.pool(x) #batchx32x12x12
    x = func.relu(self.conv3a(x))
    x = func.relu(self.conv3b(x))
    x = self.pool(x) #batchx64x6x6
    x = func.relu(self.conv4a(x))
    x = func.relu(self.conv4b(x))
    x = self.pool(x) #batchx128x3x3
    x = func.relu(self.conv5a(x))
    x = func.relu(self.conv5b(x))
    x = self.pool2(x) #batchx256x1x1

    #flatten
    x = x.view(x.size(0), -1) #batch x 256 flattened

    #fully connected layers
    x = func.relu(self.fc1(x))
    x = func.relu(self.fc2(x))
    x = self.fc3(x) #raw output softmax later

    return x

In [ ]:
#TRAINING



In [ ]:
#loss/error visualization?